In [51]:
import pandas as pd


In [52]:
data_df = pd.read_csv("data/stage-2/weather_pv_merged_and_normalised.csv")
data_df.columns

Index(['Time', 'generation(kWh)', 'power(W)', 'site', 'site_id_ttl',
       'rated_power_kw', 'datetime', 'Irradiance_Irradiance (W/m2)',
       'Rainfall_Rainfall(mm)', 'Relative_Humidity_RH (%)',
       'Sea_Level_Pressure_SLP (hPa)', 'Temperature_Temp (Degree Celsius)',
       'Visibility_Vis (km)', 'Wind_Wind Speed (m/s)',
       'Wind_Wind Direction (degree)', 'normalized_generation',
       'rated_power_w', 'normalized_power'],
      dtype='object')

In [53]:
# Sites
data_df['site'].unique()
data_df.head(5)

,Time,generation(kWh),power(W),site,site_id_ttl,rated_power_kw,datetime,Irradiance_Irradiance (W/m2),Rainfall_Rainfall(mm),Relative_Humidity_RH (%),Sea_Level_Pressure_SLP (hPa),Temperature_Temp (Degree Celsius),Visibility_Vis (km),Wind_Wind Speed (m/s),Wind_Wind Direction (degree),normalized_generation,rated_power_w,normalized_power
0,2021-08-01 00:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:00:00,6.0610,0.254,88.703,1001.79913,28.253,8.8796,1.81470,358.56,0.0,17000.0,0.0
1,2021-08-01 00:15:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:15:00,5.2896,0.000,91.764,1001.77913,27.426,15.9950,3.66970,337.83,0.0,17000.0,0.0
2,2021-08-01 00:30:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:30:00,4.8487,0.000,93.888,1001.89913,26.242,15.9950,0.87607,310.66,0.0,17000.0,0.0
3,2021-08-01 00:45:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:45:00,4.9591,0.000,95.816,1001.81913,26.850,14.0990,2.19480,173.88,0.0,17000.0,0.0
4,2021-08-01 01:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 01:00:00,5.5102,0.000,96.053,1001.76913,26.605,15.9940,1.30530,215.44,0.0,17000.0,0.0


In [54]:
data_df.isnull().sum()

Time                                 0
generation(kWh)                      0
power(W)                             0
site                                 0
site_id_ttl                          0
rated_power_kw                       0
datetime                             0
Irradiance_Irradiance (W/m2)         0
Rainfall_Rainfall(mm)                0
Relative_Humidity_RH (%)             0
Sea_Level_Pressure_SLP (hPa)         0
Temperature_Temp (Degree Celsius)    0
Visibility_Vis (km)                  0
Wind_Wind Speed (m/s)                0
Wind_Wind Direction (degree)         0
normalized_generation                0
rated_power_w                        0
normalized_power                     0
dtype: int64

In [55]:
# fixed length output size
col = 'normalized_generation'

decimal_lengths = data_df[col].astype(str).str.split('.').str[1].str.len()
decimal_lengths = decimal_lengths.fillna(0)

max_decimals = int(decimal_lengths.max())
print(max_decimals)

# Format every value to that many decimal places
data_df[col] = data_df[col].apply(lambda x: f"{x:.{max_decimals}f}")

print(data_df[col].head())

20
0    0.00000000000000000000
1    0.00000000000000000000
2    0.00000000000000000000
3    0.00000000000000000000
4    0.00000000000000000000
Name: normalized_generation, dtype: object


In [56]:
# Convert Time column to datetime (do this once)
data_df['Time'] = pd.to_datetime(data_df['Time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Check how many failed to parse
print('Failed to parse: ', end='')
print(data_df['Time'].isna().sum())

group1_sites = ['SQ Apartment25-36', 'Zone D', 'SQ Apartment13-24', 'LSK North',
       'Zone J2', 'Zone A2', 'Zone A7', 'SQ2', 'SQ Block S',
       'Indoor Sports Centre', 'SQ4', 'UG Hall8', 'UG Hall7', 'UG Hall4',
       'SQ567', 'SQ3', 'Zone A5', 'S H Ho Sports Hall', 'LSK South',]
group2_sites = ['SQ1', 'UG Hall2 RF', 'Zone A4', 'UG Hall2 2F', 'UG Hall6',
       'SQ Block P', 'Library', 'Shaw Auditorium', 'SQ Block R',
       'Wong Check She Research Centre', 'UG Hall9', 'Zone A6',
       'SQ Apartment37-48', 'UG Hall3', 'SQ Apartment1-12']

# Group 1: strictly before 2023
df_group1 = data_df[
    (data_df['site'].isin(group1_sites)) &
    (data_df['Time'].dt.year < 2023)
].copy()

# Group 2: strictly after 2023
df_group2 = data_df[
    (data_df['site'].isin(group2_sites)) &
    (data_df['Time'].dt.year > 2022)
].copy()

print(df_group1['Time'].dt.year.unique())
print(df_group2['Time'].dt.year.unique())
print(df_group1['site'].unique())
print(df_group2['site'].unique())


Failed to parse: 0
[2021 2022]
[2023]
['SQ Apartment25-36' 'Zone D' 'SQ Apartment13-24' 'LSK North' 'Zone J2'
 'Zone A2' 'Zone A7' 'SQ2' 'SQ Block S' 'Indoor Sports Centre' 'SQ4'
 'UG Hall7' 'UG Hall4' 'SQ567' 'SQ3' 'Zone A5' 'S H Ho Sports Hall'
 'LSK South']
['SQ1' 'UG Hall2 RF' 'Zone A4' 'UG Hall2 2F' 'UG Hall6' 'SQ Block P'
 'Library' 'Shaw Auditorium' 'SQ Block R' 'Wong Check She Research Centre'
 'UG Hall9' 'Zone A6' 'SQ Apartment37-48' 'UG Hall3' 'SQ Apartment1-12']


In [57]:
# Select relevant columns and rename them for readability
column_rename_map = {
    'Time': 'time',
    'Irradiance_Irradiance (W/m2)': 'irradiance_wm2',
    'Rainfall_Rainfall(mm)': 'rainfall_mm',
    'Relative_Humidity_RH (%)': 'relative_humidity_pct',
    'Sea_Level_Pressure_SLP (hPa)': 'sea_level_pressure_hpa',
    'Temperature_Temp (Degree Celsius)': 'temperature_c',
    'Visibility_Vis (km)': 'visibility_km',
    'Wind_Wind Speed (m/s)': 'wind_speed_ms',
    'normalized_generation': 'normalized_generation'
}

feature_columns = list(column_rename_map.keys())

# Select and rename columns for both groups
df_group1 = df_group1.loc[:, feature_columns].rename(columns=column_rename_map)
df_group2 = df_group2.loc[:, feature_columns].rename(columns=column_rename_map)

# Save to CSV
df_group1.to_csv('data/stage-3/training_data.csv', index=False)
df_group2.to_csv('data/stage-3/test_data.csv', index=False)

print(df_group1.columns.tolist())
print(df_group1.shape, df_group2.shape)

['time', 'irradiance_wm2', 'rainfall_mm', 'relative_humidity_pct', 'sea_level_pressure_hpa', 'temperature_c', 'visibility_km', 'wind_speed_ms', 'normalized_generation']
(857963, 9) (475570, 9)
